
## Bước 1: Chuẩn hóa annotation sang định dạng YOLO

### Mục tiêu:
1. Chuẩn hóa annotation về định dạng YOLO: `class_id x_center y_center width height` (tất cả tọa độ chuẩn hóa trong khoảng [0, 1]).
2. Không can thiệp hoặc chỉnh sửa dữ liệu gốc trong `data/raw/`.
3. Ghi nhãn đã chuẩn hóa vào `data/processed/labels/`.
4. Hỗ trợ cả 2 trường hợp:
   - Annotation gốc đã là YOLO `.txt`: validate cú pháp, kiểm tra bounds và copy/sync sang `data/processed/labels/`.
   - Annotation gốc là PASCAL VOC `.xml`: parse XML, chuẩn hóa tọa độ pixel sang tỷ lệ [0, 1] và ánh xạ nhãn lớp.
5. Duy trì cấu trúc tương đối tương thích, đảm bảo ánh xạ 1-1 giữa `image_path` và `label_path`.
6. Tự động đọc danh sách class từ `configs/base.yaml`, không hardcode class list rời rạc.
7. Log chi tiết số file xử lý, số box, phân bố theo class, và xuất báo cáo lỗi sang `results/convert_to_yolo_issues.csv` nếu phát hiện bất thường.

In [1]:
import csv
import math
import os
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image
import yaml

print("Đã import thành công các thư viện cần thiết.")


Đã import thành công các thư viện cần thiết.


In [2]:
def find_project_root() -> Path:
    """Tìm thư mục gốc của dự án chứa configs và data."""
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
CONFIG_PATH = ROOT / "configs/base.yaml"
RAW_DATA_DIR = ROOT / "data/raw/PKU-Market-PCB(Data enhanced version)"
GROUPS_PATH = ROOT / "data/splits/groups.csv"
OUTPUT_LABELS_DIR = ROOT / "data/processed/labels"
RESULTS_DIR = ROOT / "results"
ISSUES_REPORT_PATH = RESULTS_DIR / "convert_to_yolo_issues.csv"

# Đảm bảo các thư mục đích tồn tại
OUTPUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Đọc cấu hình từ configs/base.yaml
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    base_config = yaml.safe_load(f)

CLASSES = [c.strip().lower().replace(" ", "_") for c in base_config.get("classes", [])]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASSES)}
ID_TO_CLASS = {idx: name for idx, name in enumerate(CLASSES)}

print(f"Project root: {ROOT}")
print(f"Dataset gốc: {RAW_DATA_DIR.relative_to(ROOT)}")
print(f"Thư mục nhãn đích: {OUTPUT_LABELS_DIR.relative_to(ROOT)}")
print(f"Danh sách {len(CLASSES)} classes từ configs/base.yaml: {CLASSES}")
print(f"Class to ID mapping: {CLASS_TO_ID}")


Project root: /Users/Project/kltn-pcb
Dataset gốc: data/raw/PKU-Market-PCB(Data enhanced version)
Thư mục nhãn đích: data/processed/labels
Danh sách 6 classes từ configs/base.yaml: ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
Class to ID mapping: {'missing_hole': 0, 'mouse_bite': 1, 'open_circuit': 2, 'short': 3, 'spur': 4, 'spurious_copper': 5}


In [3]:
IMAGE_SUFFIXES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
CSV_COLUMNS = [
    "issue_type",
    "image_path",
    "source_label_path",
    "line_number",
    "class_id",
    "details",
]

def log_issue(
    issues_list: list[dict],
    issue_type: str,
    image_path: Path | None = None,
    source_label_path: Path | None = None,
    line_number: int | str = "",
    class_id: int | str = "",
    details: str = "",
) -> None:
    def _rel(p: Path | None) -> str:
        if p is None:
            return ""
        try:
            return str(p.relative_to(ROOT))
        except ValueError:
            return str(p)

    issues_list.append({
        "issue_type": issue_type,
        "image_path": _rel(image_path),
        "source_label_path": _rel(source_label_path),
        "line_number": line_number,
        "class_id": class_id,
        "details": details,
    })

def find_source_label_for_image(image_path: Path, raw_dataset_dir: Path) -> Path | None:
    """Tìm file nhãn nguồn (.txt hoặc .xml) cho một ảnh."""
    try:
        relative = image_path.relative_to(raw_dataset_dir)
    except ValueError:
        return None

    for ext in [".txt", ".xml"]:
        if "images" in relative.parts:
            idx = relative.parts.index("images")
            labels_dir = raw_dataset_dir.joinpath(*relative.parts[:idx], "labels")
            mirrored_path = labels_dir.joinpath(*relative.parts[idx + 1:]).with_suffix(ext)
            flattened_path = labels_dir / image_path.with_suffix(ext).name
            if mirrored_path.is_file():
                return mirrored_path
            if flattened_path.is_file():
                return flattened_path
        else:
            direct = image_path.with_suffix(ext)
            if direct.is_file():
                return direct
    return None

def map_image_to_processed_label(image_path: Path, raw_dataset_dir: Path, processed_labels_dir: Path) -> Path:
    """Xác định đường dẫn nhãn trong data/processed/labels/ tương ứng với image_path."""
    try:
        rel = image_path.relative_to(raw_dataset_dir)
        parts = list(rel.parts)
        if "images" in parts:
            parts.remove("images")
        return processed_labels_dir.joinpath(*parts).with_suffix(".txt")
    except ValueError:
        return processed_labels_dir / image_path.with_suffix(".txt").name

def validate_and_normalize_yolo_content(
    content: str,
    source_path: Path,
    image_path: Path,
    num_classes: int,
    issues_list: list[dict],
) -> list[str]:
    """Validate các dòng annotation YOLO và trả về danh sách các dòng chuẩn hóa."""
    normalized_lines: list[str] = []
    seen_boxes: set[tuple[int, float, float, float, float]] = set()

    lines = content.strip().splitlines()
    for line_idx, line in enumerate(lines, start=1):
        line_clean = line.strip()
        if not line_clean:
            continue

        parts = line_clean.split()
        if len(parts) != 5:
            log_issue(issues_list, "malformed_line", image_path, source_path, line_idx, "", f"YOLO format yêu cầu đúng 5 giá trị, nhận được: {len(parts)}")
            continue

        try:
            class_id = int(parts[0])
            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])
        except ValueError:
            log_issue(issues_list, "invalid_type", image_path, source_path, line_idx, parts[0], "class_id phải là int, tọa độ phải là float")
            continue

        if not (0 <= class_id < num_classes):
            log_issue(issues_list, "class_id_out_of_range", image_path, source_path, line_idx, class_id, f"class_id {class_id} không thuộc [0, {num_classes - 1}]")
            continue

        if not (math.isfinite(xc) and math.isfinite(yc) and 0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0):
            log_issue(issues_list, "center_out_of_range", image_path, source_path, line_idx, class_id, f"Tọa độ tâm ({xc}, {yc}) ngoài khoảng [0, 1]")
            continue

        if not (math.isfinite(w) and math.isfinite(h) and w > 0.0 and h > 0.0 and w <= 1.0 and h <= 1.0):
            log_issue(issues_list, "dimension_out_of_range", image_path, source_path, line_idx, class_id, f"Kích thước w={w}, h={h} không hợp lệ")
            continue

        xmin, xmax = xc - w / 2.0, xc + w / 2.0
        ymin, ymax = yc - h / 2.0, yc + h / 2.0
        if xmin < 0.0 or xmax > 1.0 or ymin < 0.0 or ymax > 1.0:
            log_issue(issues_list, "box_outside_boundary", image_path, source_path, line_idx, class_id, f"Box tràn biên ảnh: [{xmin:.4f}, {ymin:.4f}, {xmax:.4f}, {ymax:.4f}]")

        box_key = (class_id, round(xc, 6), round(yc, 6), round(w, 6), round(h, 6))
        if box_key in seen_boxes:
            log_issue(issues_list, "duplicate_box", image_path, source_path, line_idx, class_id, "Box trùng lặp với box trước đó trong cùng file")
        seen_boxes.add(box_key)

        normalized_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    return normalized_lines

def convert_voc_xml_content(
    xml_path: Path,
    image_path: Path,
    class_to_id: dict[str, int],
    issues_list: list[dict],
) -> list[str]:
    """Chuyển đổi file XML Pascal VOC sang các dòng YOLO chuẩn hóa."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        log_issue(issues_list, "xml_parse_error", image_path, xml_path, "", "", f"Lỗi parse XML: {e}")
        return []

    size_tag = root.find("size")
    img_w, img_h = 0.0, 0.0
    if size_tag is not None:
        w_elem = size_tag.find("width")
        h_elem = size_tag.find("height")
        if w_elem is not None and h_elem is not None:
            try:
                img_w = float(w_elem.text)
                img_h = float(h_elem.text)
            except (ValueError, TypeError):
                pass

    if img_w <= 0 or img_h <= 0:
        if image_path.is_file():
            try:
                with Image.open(image_path) as img:
                    img_w, img_h = float(img.width), float(img.height)
            except Exception as e:
                log_issue(issues_list, "cannot_read_image_size", image_path, xml_path, "", "", f"Không đọc được kích thước ảnh: {e}")
                return []
        else:
            log_issue(issues_list, "missing_image_for_xml", image_path, xml_path, "", "", "XML thiếu size và không tìm thấy ảnh để fallback")
            return []

    normalized_lines: list[str] = []
    for obj_idx, obj in enumerate(root.findall("object"), start=1):
        name_elem = obj.find("name")
        if name_elem is None or not name_elem.text:
            log_issue(issues_list, "missing_object_name", image_path, xml_path, obj_idx, "", "Đối tượng thiếu thẻ <name>")
            continue

        cls_name = name_elem.text.strip().lower().replace(" ", "_")
        if cls_name not in class_to_id:
            log_issue(issues_list, "unknown_class_name", image_path, xml_path, obj_idx, cls_name, f"Lớp '{cls_name}' không nằm trong configs/base.yaml")
            continue

        class_id = class_to_id[cls_name]
        bndbox = obj.find("bndbox")
        if bndbox is None:
            log_issue(issues_list, "missing_bndbox", image_path, xml_path, obj_idx, class_id, "Đối tượng thiếu thẻ <bndbox>")
            continue

        try:
            xmin = float(bndbox.find("xmin").text)
            ymin = float(bndbox.find("ymin").text)
            xmax = float(bndbox.find("xmax").text)
            ymax = float(bndbox.find("ymax").text)
        except (ValueError, TypeError, AttributeError) as e:
            log_issue(issues_list, "invalid_bndbox_coords", image_path, xml_path, obj_idx, class_id, f"Tọa độ bndbox không hợp lệ: {e}")
            continue

        if xmax <= xmin or ymax <= ymin:
            log_issue(issues_list, "inverted_bndbox", image_path, xml_path, obj_idx, class_id, f"Tọa độ đảo ngược: [{xmin}, {ymin}, {xmax}, {ymax}]")
            continue

        xc = ((xmin + xmax) / 2.0) / img_w
        yc = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        xc = max(0.0, min(1.0, xc))
        yc = max(0.0, min(1.0, yc))
        w = max(0.0, min(1.0, w))
        h = max(0.0, min(1.0, h))

        normalized_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    return normalized_lines


In [4]:
# Thực hiện chuẩn hóa và đồng bộ sang data/processed/labels/
issues_list: list[dict] = []

if GROUPS_PATH.is_file():
    groups_df = pd.read_csv(GROUPS_PATH)
    image_paths = [ROOT / p for p in groups_df["image_path"]]
    print(f"Đọc {len(image_paths)} đường dẫn ảnh từ {GROUPS_PATH.name}.")
else:
    image_paths = sorted(p for p in RAW_DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    print(f"Quét trực tiếp {len(image_paths)} ảnh từ {RAW_DATA_DIR.name}.")

stats = {
    "total_images": len(image_paths),
    "labels_written": 0,
    "total_boxes": 0,
    "boxes_per_class": {c: 0 for c in CLASSES},
    "images_without_label": 0,
    "format_yolo_txt": 0,
    "format_voc_xml": 0,
}

for img_p in image_paths:
    src_lbl = find_source_label_for_image(img_p, RAW_DATA_DIR)
    dst_lbl = map_image_to_processed_label(img_p, RAW_DATA_DIR, OUTPUT_LABELS_DIR)
    dst_lbl.parent.mkdir(parents=True, exist_ok=True)

    if src_lbl is None or not src_lbl.is_file():
        stats["images_without_label"] += 1
        dst_lbl.write_text("", encoding="utf-8")
        stats["labels_written"] += 1
        continue

    if src_lbl.suffix.lower() == ".xml":
        stats["format_voc_xml"] += 1
        lines = convert_voc_xml_content(src_lbl, img_p, CLASS_TO_ID, issues_list)
    else:
        stats["format_yolo_txt"] += 1
        raw_content = src_lbl.read_text(encoding="utf-8")
        lines = validate_and_normalize_yolo_content(raw_content, src_lbl, img_p, len(CLASSES), issues_list)

    dst_lbl.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    stats["labels_written"] += 1
    stats["total_boxes"] += len(lines)

    for line in lines:
        cid = int(line.split()[0])
        stats["boxes_per_class"][CLASSES[cid]] += 1

# Ghi báo cáo lỗi
with open(ISSUES_REPORT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    writer.writerows(issues_list)

print(f"Đã hoàn thành chuẩn hóa! Ghi {stats['labels_written']} file nhãn vào {OUTPUT_LABELS_DIR.relative_to(ROOT)}.")
print(f"Tổng số lỗi/cảnh báo: {len(issues_list)} (chi tiết tại {ISSUES_REPORT_PATH.relative_to(ROOT)})")


Đọc 4837 đường dẫn ảnh từ groups.csv.


Đã hoàn thành chuẩn hóa! Ghi 4837 file nhãn vào data/processed/labels.
Tổng số lỗi/cảnh báo: 0 (chi tiết tại results/convert_to_yolo_issues.csv)


In [5]:
# Tổng kết thống kê chi tiết
print("=" * 60)
print("BÁO CÁO TỔNG KẾT CHUẨN HÓA ANNOTATION (TUẦN 4 - BƯỚC 1)")
print("=" * 60)
print(f"Tổng số ảnh xử lý             : {stats['total_images']:,}")
print(f"Tổng số file label được tạo   : {stats['labels_written']:,}")
print(f"Số file từ nguồn YOLO .txt    : {stats['format_yolo_txt']:,}")
print(f"Số file từ nguồn VOC .xml     : {stats['format_voc_xml']:,}")
print(f"Số ảnh không có nhãn (trống)  : {stats['images_without_label']:,}")
print(f"Tổng số bounding boxes        : {stats['total_boxes']:,}")
print(f"Số vấn đề/cảnh báo phát hiện  : {len(issues_list)}")
print("-" * 60)
print("Phân bố bounding box theo từng lớp (6 classes):")
for idx, cname in enumerate(CLASSES):
    cnt = stats["boxes_per_class"][cname]
    pct = (cnt / stats["total_boxes"] * 100) if stats["total_boxes"] > 0 else 0
    print(f"  [{idx}] {cname:<18}: {cnt:>6,} boxes ({pct:5.2f}%)")
print("=" * 60)


BÁO CÁO TỔNG KẾT CHUẨN HÓA ANNOTATION (TUẦN 4 - BƯỚC 1)
Tổng số ảnh xử lý             : 4,837
Tổng số file label được tạo   : 4,837
Số file từ nguồn YOLO .txt    : 4,837
Số file từ nguồn VOC .xml     : 0
Số ảnh không có nhãn (trống)  : 0
Tổng số bounding boxes        : 19,003
Số vấn đề/cảnh báo phát hiện  : 0
------------------------------------------------------------
Phân bố bounding box theo từng lớp (6 classes):
  [0] missing_hole      :  3,306 boxes (17.40%)
  [1] mouse_bite        :  3,274 boxes (17.23%)
  [2] open_circuit      :  3,138 boxes (16.51%)
  [3] short             :  3,105 boxes (16.34%)
  [4] spur              :  3,038 boxes (15.99%)
  [5] spurious_copper   :  3,142 boxes (16.53%)


In [6]:
# Kiểm tra đối chứng (spot-check) một vài file label output
sample_paths = [
    OUTPUT_LABELS_DIR / "train/01_missing_hole_01.txt",
    OUTPUT_LABELS_DIR / "train/train1/01_missing_hole_01.txt",
    OUTPUT_LABELS_DIR / "valid/04_open_circuit_02_1.txt",
    OUTPUT_LABELS_DIR / "test/01_missing_hole_01_1.txt",
]

print("KIỂM TRA ĐỐI CHỨNG CÁC FILE OUTPUT:")
print("-" * 60)
for sp in sample_paths:
    print(f"File: {sp.relative_to(ROOT)}")
    if sp.is_file():
        lines = sp.read_text(encoding="utf-8").strip().splitlines()
        print(f"  Tồn tại: True | Số boxes: {len(lines)}")
        for line in lines[:3]:
            parts = line.split()
            cid = int(parts[0])
            print(f"    Class {cid} ({CLASSES[cid]}): {' '.join(parts[1:])}")
        if len(lines) > 3:
            print(f"    ... (+{len(lines) - 3} boxes)")
    else:
        print("  Tồn tại: FALSE (Không tìm thấy file!)")
    print("-" * 60)


KIỂM TRA ĐỐI CHỨNG CÁC FILE OUTPUT:
------------------------------------------------------------
File: data/processed/labels/train/01_missing_hole_01.txt
  Tồn tại: True | Số boxes: 3
    Class 0 (missing_hole): 0.822182 0.820618 0.023401 0.034678
    Class 0 (missing_hole): 0.542518 0.230139 0.021753 0.039092
    Class 0 (missing_hole): 0.580587 0.519546 0.023401 0.037831
------------------------------------------------------------
File: data/processed/labels/train/train1/01_missing_hole_01.txt
  Tồn tại: True | Số boxes: 3
    Class 0 (missing_hole): 0.822182 0.820618 0.023401 0.034678
    Class 0 (missing_hole): 0.542518 0.230139 0.021753 0.039092
    Class 0 (missing_hole): 0.580587 0.519546 0.023401 0.037831
------------------------------------------------------------
File: data/processed/labels/valid/04_open_circuit_02_1.txt
  Tồn tại: True | Số boxes: 3
    Class 2 (open_circuit): 0.427683 0.854302 0.026178 0.031656
    Class 2 (open_circuit): 0.413122 0.641843 0.026505 0.023133